# Обучение LSTM-модели на хронологической выборке

## Импортирование библиотек и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from pathlib import Path
import joblib

OUTPUT_DIR = Path("../data/processed")

X_train = np.load(OUTPUT_DIR / "X_train_r2.npy")
y_train = np.load(OUTPUT_DIR / "y_train_r2.npy")
X_test = np.load(OUTPUT_DIR / "X_test_r2.npy")
y_test = np.load(OUTPUT_DIR / "y_test_r2.npy")
scaler = joblib.load(OUTPUT_DIR / "scaler_r2.joblib")

WINDOW_SIZE = 12
HIDDEN_SIZE = 50

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print(f"Размерность X_train: {X_train_t.shape}")
print(f"Размерность X_test:  {X_test_t.shape}")

## Архитектура нейросети

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :]) 
        return out

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
            self.counter = 0

## Настройка и обучение

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(input_size=1, hidden_size=HIDDEN_SIZE).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
early_stopping = EarlyStopping(patience=15)

val_size = int(len(X_train_t) * 0.2)
train_size = len(X_train_t) - val_size

X_tr, X_val = X_train_t[:train_size], X_train_t[train_size:]
y_tr, y_val = y_train_t[:train_size], y_train_t[train_size:]

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=16, shuffle=False)

epochs = 1000
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * X_batch.size(0)
    
    epoch_train_loss /= len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            epoch_val_loss += loss.item() * X_batch.size(0)
    
    epoch_val_loss /= len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    early_stopping(epoch_val_loss, model)
    if early_stopping.early_stop:
        print(f"Ранняя остановка на эпохе {epoch}")
        break

model.load_state_dict(early_stopping.best_model_state)
print("Обучение завершено.")

## Оценка качества на тестовой выборке

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t.to(device)).cpu().numpy()

mse = mean_squared_error(y_test, y_pred_scaled)
mae = mean_absolute_error(y_test, y_pred_scaled)
r2 = r2_score(y_test, y_pred_scaled)

print(f"MSE: {mse:.6f}")
print(f"MAE: {mae:.6f}")
print(f"R^2: {r2 * 100:.2f}%")

## Рекурсивный прогноз на апрель - декабрь 2022

In [ ]:
df_full = pd.read_csv(OUTPUT_DIR / "cpi_monthly.csv", parse_dates=["Date"])
last_real_cpi = df_full[df_full['Date'] <= '2022-03-01']['cpi_mom'].values
scaled_last_window = scaler.transform(last_real_cpi.reshape(-1, 1))

current_window = scaled_last_window[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
current_window_t = torch.tensor(current_window, dtype=torch.float32).to(device)
predictions_scaled = []

model.eval()
with torch.no_grad():
    for step in range(9):
        next_pred_scaled = model(current_window_t).cpu().numpy()[0, 0]
        predictions_scaled.append(next_pred_scaled)
        
        next_pred_tensor = torch.tensor([[[next_pred_scaled]]], dtype=torch.float32).to(device)
        current_window_t = torch.cat((current_window_t[:, 1:, :], next_pred_tensor), dim=1)

predictions_real = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1, 1))[:, 0]

months = ['Апрель', 'Май', 'Июнь', 'Июль', 'Август', 'Сентябрь', 'Октябрь', 'Ноябрь', 'Декабрь']
for month, val in zip(months, predictions_real):
    print(f"{month} 2022: {val:.2f}%")

## Расчёт годовой инфляции за 2022 год

In [ ]:
actual_q1_2022 = df_full[(df_full['Date'] >= '2022-01-01') & (df_full['Date'] <= '2022-03-01')]['cpi_mom'].values

full_year_2022_mom = np.concatenate([actual_q1_2022, predictions_real])
annual_cpi = np.prod(full_year_2022_mom / 100)
annual_inflation = (annual_cpi - 1) * 100

print(f"Прогноз годовой инфляции на 2022 год: {annual_inflation:.2f}%")
print(f"В статье: 20.00%")
print(f"Реальная инфляция: 11.94%")

## Визуализация прогноза

In [ ]:
forecast_dates = pd.date_range(start='2022-04-01', periods=9, freq='MS')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_full['Date'].tail(36), df_full['cpi_mom'].tail(36), label='Фактические значения', color='steelblue')
ax.plot(forecast_dates, predictions_real, label='Прогноз LSTM', color='green', marker='o', linestyle='--')

ax.set_title('Прогноз ИПЦ на 9 месяцев вперед')
ax.set_xlabel('Дата')
ax.set_ylabel('cpi_mom, %')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## Прогноз на 2022 год без учета марта

In [ ]:
df_full = pd.read_csv(OUTPUT_DIR / "cpi_monthly.csv", parse_dates=["Date"])

last_2021_cpi = df_full[df_full['Date'] <= '2021-12-01']['cpi_mom'].values
scaled_2021_window = scaler.transform(last_2021_cpi.reshape(-1, 1))

current_window_2021 = scaled_2021_window[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
current_window_2021_t = torch.tensor(current_window_2021, dtype=torch.float32).to(device)

predictions_blind_scaled = []

model.eval()
with torch.no_grad():
    for step in range(12):
        next_pred_scaled = model(current_window_2021_t).cpu().numpy()[0, 0]
        predictions_blind_scaled.append(next_pred_scaled)
        
        next_pred_tensor = torch.tensor([[[next_pred_scaled]]], dtype=torch.float32).to(device)
        current_window_2021_t = torch.cat((current_window_2021_t[:, 1:, :], next_pred_tensor), dim=1)

predictions_blind_real = scaler.inverse_transform(np.array(predictions_blind_scaled).reshape(-1, 1))[:, 0]

months_full = ['Январь', 'Февраль', 'Март', 'Апрель', 'Май', 'Июнь', 'Июль', 'Август', 'Сентябрь', 'Октябрь', 'Ноябрь', 'Декабрь']
print("Слепой прогноз на 2022 год без марта:")
for month, val in zip(months_full, predictions_blind_real):
    print(f"{month} 2022: {val:.2f}%")

In [ ]:
annual_cpi_blind = np.prod(predictions_blind_real / 100)
annual_inflation_blind = (annual_cpi_blind - 1) * 100

print(f"Слепой прогноз годовой инфляции на 2022 год: {annual_inflation_blind:.2f}%")
print(f"Реальная инфляция: 11.94%")

In [ ]:
forecast_dates_blind = pd.date_range(start='2022-01-01', periods=12, freq='MS')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_full['Date'].tail(24), df_full['cpi_mom'].tail(24), label='Фактические значения', color='steelblue')
ax.plot(forecast_dates_blind, predictions_blind_real, label='Слепой прогноз LSTM', color='orange', marker='o', linestyle='--')

ax.set_title('Слепой прогноз ИПЦ на 2022 год')
ax.set_xlabel('Дата')
ax.set_ylabel('cpi_mom, %')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()